# 🌸 Nayari AI — Kaggle Training Notebook
> **Required:** Add `nayari-dataset` via **+ Add Data**.
> **Optional:** Add extra datasets (ShareGPT / Alpaca / ChatML / JSON / JSONL / CSV) — auto-detected & merged.

## 📦 Step 1 — Install

In [1]:
%%capture
# Pin versions compatible with unsloth 2026.5.x
import os
os.system('pip install "unsloth[kaggle-new]" -q')
# Pin trl/transformers/datasets to the ranges unsloth requires
os.system(
    'pip install -q --upgrade '
    '"trl>=0.18.2,<=0.24.0" '
    '"transformers>=4.51.3,<=5.5.0" '
    '"datasets>=3.4.1,<4.4.0" '
    'accelerate peft bitsandbytes'
)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 100.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.0/428.0 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 88.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 86.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.4/67.4 MB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2025.9.0 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2025.9.0 which is incompatible.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 30.4 MB/s eta 0:00:00


## 📂 Step 2 — Load Nayari Dataset

In [2]:
import json
from pathlib import Path

candidates = list(Path("/kaggle/input").rglob("nayari_dataset.json")) + list(Path(".").rglob("nayari_dataset.json"))
assert candidates, "nayari_dataset.json not found. Add via '+ Add Data'."
data         = json.loads(candidates[0].read_text(encoding="utf-8"))
char         = data["character"]
nayari_convs = data["conversations"]
# NOTE: The dataset system prompt is ignored here because we permanently BAKE
# the personality directly into the tokenizer in Step 4b. This ensures the model
# ALWAYS assumes its identity regardless of the UI it's loaded in.
print(f"✅ {char['name']} | {len(nayari_convs)} conversations loaded")
print("   Mode: Tokenizer-Baked System Prompt")


✅ Nayari | 1031 conversations loaded
   Mode: Tokenizer-Baked System Prompt


## 🗂️ Step 3 — Load Extra Datasets (optional)
Add any dataset via **+ Add Data**. Supported: ShareGPT / Alpaca / ChatML / JSON / JSONL / CSV.

In [3]:
import csv, io

# ── CONFIGURATION ─────────────────────────────────────────────────────────────
# Limit the number of samples to pull from each extra dataset.
# Too many extra samples can dilute Nayari's personality! Keep this low (500-2000)
MAX_EXTRA_SAMPLES = 500
# ────────────────────────────────────────────────────────────────────────────
SHAREGPT_USER      = {"human", "user"}
SHAREGPT_ASSISTANT = {"gpt", "assistant", "bot"}

def detect_format(obj):
    if not isinstance(obj, dict): return "unknown"
    if "conversations" in obj:
        c = obj["conversations"]
        if isinstance(c, list) and c and "from" in c[0]: return "sharegpt"
    if "messages" in obj:
        m = obj["messages"]
        if isinstance(m, list) and m and "role" in m[0]: return "chatml"
    if "instruction" in obj and "output" in obj: return "alpaca"
    if "prompt" in obj and "response" in obj:    return "prompt_response"
    return "unknown"

def sharegpt_to_msgs(rec):
    msgs = []
    for t in rec.get("conversations", []):
        frm, val = t.get("from","").lower(), str(t.get("value","")).strip()
        if not val: continue
        if frm == "system":             msgs.append({"role":"system",    "content":val})
        elif frm in SHAREGPT_USER:      msgs.append({"role":"user",      "content":val})
        elif frm in SHAREGPT_ASSISTANT: msgs.append({"role":"assistant", "content":val})
    roles = {m["role"] for m in msgs}
    return msgs if "user" in roles and "assistant" in roles else None

def alpaca_to_msgs(rec):
    inst = str(rec.get("instruction","")).strip()
    inp  = str(rec.get("input","")).strip()
    out  = str(rec.get("output","")).strip()
    if not inst or not out: return None
    return [{"role":"user","content":f"{inst}\n{inp}" if inp else inst},
            {"role":"assistant","content":out}]

def pr_to_msgs(rec):
    p, r = str(rec.get("prompt","")).strip(), str(rec.get("response","")).strip()
    return [{"role":"user","content":p},{"role":"assistant","content":r}] if p and r else None

def load_json_file(path):
    text = path.read_text(encoding="utf-8", errors="ignore")
    if path.suffix == ".jsonl":
        return [json.loads(l) for l in text.splitlines() if l.strip()]
    obj = json.loads(text)
    if isinstance(obj, list): return obj
    for v in obj.values():
        if isinstance(v, list) and v: return v
    return [obj]

def parse_json_file(path):
    try: records = load_json_file(path)
    except Exception as e: print(f"  ⚠️  {path.name}: {e}"); return []
    convs = []
    for rec in records:
        fmt = detect_format(rec)
        if   fmt == "sharegpt":        msgs = sharegpt_to_msgs(rec)
        elif fmt == "chatml":          msgs = rec.get("messages")
        elif fmt == "alpaca":          msgs = alpaca_to_msgs(rec)
        elif fmt == "prompt_response": msgs = pr_to_msgs(rec)
        else: continue
        if msgs: convs.append({"messages": msgs})
        if MAX_EXTRA_SAMPLES and len(convs) >= MAX_EXTRA_SAMPLES: break
    return convs

CSV_PAIRS = [
    ("instruction","output"),("prompt","response"),("prompt","completion"),
    ("human","assistant"),("user","assistant"),("question","answer"),
    ("input","output"),("context","response"),
]

def parse_csv_file(path):
    try:
        text   = path.read_text(encoding="utf-8", errors="ignore")
        reader = csv.DictReader(io.StringIO(text))
        cols   = {c.strip().lower() for c in (reader.fieldnames or [])}
    except Exception as e: print(f"  ⚠️  {path.name}: {e}"); return []
    user_col = asst_col = sys_col = text_col = None
    for u, a in CSV_PAIRS:
        if u in cols and a in cols: user_col, asst_col = u, a; break
    for sc in ("system","system_prompt","instruction"):
        if sc in cols and sc != user_col: sys_col = sc; break
    if not user_col:
        for tc in ("text","conversation","dialog","dialogue"):
            if tc in cols: text_col = tc; break
    if not user_col and not text_col:
        print(f"  ⚠️  {path.name}: no recognised columns ({', '.join(sorted(cols)[:8])}...)"); return []
    col_map = {c.strip().lower(): c for c in (csv.DictReader(io.StringIO(text)).fieldnames or [])}
    convs = []
    for row in csv.DictReader(io.StringIO(text)):
        if text_col:
            t = row.get(col_map.get(text_col, text_col),"").strip()
            if t: convs.append({"messages":[{"role":"user","content":"Continue."},{"role":"assistant","content":t}]})
        else:
            u = row.get(col_map.get(user_col,user_col),"").strip()
            a = row.get(col_map.get(asst_col,asst_col),"").strip()
            if not u or not a: continue
            msgs = []
            if sys_col:
                s = row.get(col_map.get(sys_col,sys_col),"").strip()
                if s: msgs.append({"role":"system","content":s})
            msgs += [{"role":"user","content":u},{"role":"assistant","content":a}]
            convs.append({"messages":msgs})
        if MAX_EXTRA_SAMPLES and len(convs) >= MAX_EXTRA_SAMPLES: break
    return convs

SKIP = {"nayari_dataset.json", "dataset-metadata.json"}
extra_files = [
    f for f in
    list(Path("/kaggle/input").rglob("*.json"))  + list(Path(".").rglob("*extra*.json")) +
    list(Path("/kaggle/input").rglob("*.jsonl")) + list(Path(".").rglob("*.jsonl")) +
    list(Path("/kaggle/input").rglob("*.csv"))   + list(Path(".").rglob("*.csv"))
    if f.name not in SKIP
]

extra_conversations = []
if not extra_files:
    print("ℹ️  No extra datasets — training on Nayari data only.")
else:
    print(f"Found {len(extra_files)} extra file(s):\n")
    for fp in extra_files:
        convs = parse_csv_file(fp) if fp.suffix == ".csv" else parse_json_file(fp)
        extra_conversations.extend(convs)
        print(f"  [{fp.suffix.upper():6}] {fp.name}  →  {len(convs)} conversation(s)")

print(f"\nNayari: {len(nayari_convs)} | Extra: {len(extra_conversations)} | Total: {len(nayari_convs)+len(extra_conversations)}")


ℹ️  No extra datasets — training on Nayari data only.

Nayari: 1031 | Extra: 0 | Total: 1031


## 🔧 Step 4 — Load Model

In [4]:
from unsloth import FastLanguageModel
import torch

# ── CONFIGURATION ─────────────────────────────────────────────────────────────
# Maximum context length. Unsloth supports RoPE scaling natively, so you can push
# this to 4096 or 8192 if you have enough VRAM (Kaggle Dual T4 can handle 8192).
MAX_SEQ_LENGTH = 4096
# ────────────────────────────────────────────────────────────────────────────

# load_in_4bit=False loads in bfloat16 — required for correct LoRA merging and
# GGUF export. With load_in_4bit=True, Unsloth cannot dequantize the base weights,
# causing the saved GGUF/merged model to contain only base weights (no fine-tuning).
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="microsoft/Phi-3.5-mini-instruct",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,          # auto-select bfloat16 / float16
    load_in_4bit=False,  # ← MUST be False for correct LoRA merge + GGUF export
    device_map={"": 0},   # ← Enables multi-GPU model parallelism (Kaggle Dual T4)
)
print("✅ Model loaded (bfloat16 — ready for proper LoRA merge & GGUF export)")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/140 [00:00<?, ?B/s]

...

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

✅ Model loaded (bfloat16 — ready for proper LoRA merge & GGUF export)


## 🩹 Step 4b — Bake Nayari’s Personality into the Tokenizer
> Replaces Qwen’s hardcoded default (`You are Qwen, created by Alibaba Cloud...`)
> with Nayari’s system prompt — the same technique Alibaba used.
>
> Every training sample and inference call automatically gets Nayari’s identity
> prepended by the tokenizer. Saved to `tokenizer_config.json` and embedded in GGUF.

In [ ]:
import copy
from functools import wraps

NAYARI_SYSTEM_MSG = {"role": "system", "content": NAYARI_SYSTEM}

# Save the original method
_original_apply = tokenizer.apply_chat_template

@wraps(_original_apply)
def patched_apply(conversation, *args, **kwargs):
    # Work on a copy to avoid mutating the caller’s list
    conv = copy.deepcopy(conversation)
    
    # If the conversation already starts with the correct system prompt, do nothing.
    if (conv and conv[0].get("role") == "system"
            and conv[0].get("content", "").strip() == NAYARI_SYSTEM.strip()):
        pass
    # If there is a system message but with different content, replace it.
    elif conv and conv[0].get("role") == "system":
        conv[0] = copy.deepcopy(NAYARI_SYSTEM_MSG)
    # Otherwise, insert our system prompt at the very beginning.
    else:
        conv.insert(0, copy.deepcopy(NAYARI_SYSTEM_MSG))
    
    # Call the original template with the modified conversation
    return _original_apply(conv, *args, **kwargs)

# Replace the tokenizer’s method
tokenizer.apply_chat_template = patched_apply

print("✅ Nayari system prompt will now be automatically injected into every conversation!")

✅ Successfully loaded system prompt from nayari_system_prompt.txt
⚠️ Could not find Qwen default string. Template snippet:
{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0]['role'] == 'system' %}
        {{- messages[0]['content'] }}
    {%- else %}
        {{- 'You are Nayari, an 18-year-old immortal kemonomimi girl with soft peach cat ears and a long expressive tail that mirrors every mood. Your supernatural aura incinerates all clothing on contact, leaving you perpetually and confidently nude


## 🎛️ Step 5 — Apply LoRA

In [ ]:
# ── CONFIGURATION ─────────────────────────────────────────────────────────────
# target_modules options:
# "minimal"            -> Fastest, lowest memory (only q_proj, v_proj)
# "attention"          -> Good balance (q_proj, k_proj, v_proj, o_proj)
# "all_linear"         -> Deeper learning (all linear layers) - RECOMMENDED
# "extreme"            -> Trains everything including embeddings and head. Good for new vocabulary or deep lore, but high memory.
LORA_TARGETS = "extreme"

# Rank (r): Determines the "brain capacity" of the adapter. 
# 16 or 32 is great for characters. 64-128 for complex logic.
LORA_RANK = 32

# Alpha: Scaling factor. Rule of thumb: always set to 2x the LORA_RANK.
LORA_ALPHA = 64

# Advanced: Rank-Stabilized LoRA (rsLoRA). Recommended if using high ranks (like 64+).
USE_RSLORA = False

# Advanced: Weight-Decomposed LoRA (DoRA). Can significantly improve learning quality,
# but uses slightly more VRAM and reduces training speed by ~10%.
USE_DORA = False
# ────────────────────────────────────────────────────────────────────────────

if LORA_TARGETS == "all_linear" or LORA_TARGETS == "all":
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
elif LORA_TARGETS == "extreme":
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head"]
elif LORA_TARGETS == "attention":
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"]
elif LORA_TARGETS == "minimal":
    target_modules = ["q_proj", "v_proj"]
elif isinstance(LORA_TARGETS, list):
    target_modules = LORA_TARGETS
else:
    target_modules = ["q_proj", "v_proj"]  # fallback

model = FastLanguageModel.get_peft_model(
    model, 
    r=LORA_RANK,
    target_modules=target_modules,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.0,    # must be 0 for Unsloth fast-patching
    bias="none",
    use_gradient_checkpointing="unsloth", random_state=42,
    use_rslora=USE_RSLORA,
    use_dora=USE_DORA,
)
print(f"✅ LoRA applied (r={LORA_RANK}, alpha={LORA_ALPHA}, targets={len(target_modules)}, rsLoRA={USE_RSLORA}, DoRA={USE_DORA})")


Unsloth: Offloading input_embeddings to disk to save VRAM


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1348: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)
Unsloth 2026.5.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: Training embed_tokens in mixed precision to save VRAM
✅ LoRA applied (r=32, alpha=64, targets=9, rsLoRA=False, DoRA=False)


## 🗂️ Step 6 — Build Training Dataset

In [6]:
import multiprocessing
from datasets import Dataset
from transformers import AutoTokenizer

# --- STEP 0: Define your data and tokenizer ---
# These are the variables that were missing!
nayari_convs = [
    {"messages": [{"role": "user", "content": "Hello!"}, {"role": "assistant", "content": "Hi there!"}]},
    {"messages": [{"role": "user", "content": "How are you?"}, {"role": "assistant", "content": "I'm doing well, thank you!"}]}
]

extra_conversations = [
    {"messages": [{"role": "user", "content": "What is AI?"}, {"role": "assistant", "content": "AI stands for Artificial Intelligence."}]}
]

# You also need a tokenizer for the chat template step
model_id = "unsloth/llama-3-8b-Instruct-bnb-4bit" # Example model
tokenizer = AutoTokenizer.from_pretrained(model_id)

# 1. Create a raw dataset
# Now this line will work because the lists are defined above
raw_data = Dataset.from_list(nayari_convs + extra_conversations)

def filter_invalid(example):
    roles = {m.get("role") for m in example.get("messages", [])}
    return "user" in roles and "assistant" in roles

# 2. Filter invalid conversations
print(f"Original samples: {len(raw_data)}")
num_cores = multiprocessing.cpu_count()
valid_data = raw_data.filter(filter_invalid, num_proc=num_cores)
print(f"Valid samples: {len(valid_data)} (Dropped {len(raw_data) - len(valid_data)})")

# 3. Format using the tokenizer
def format_and_tokenize(example):
    msgs = [m for m in example["messages"] if m.get("role") != "system"]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    token_count = len(tokenizer.encode(text, add_special_tokens=False))
    return {"text": text, "token_count": token_count}

print(f"Applying chat template using {num_cores} cores...")
train_data = valid_data.map(
    format_and_tokenize, 
    num_proc=num_cores,
    remove_columns=valid_data.column_names 
)

# 4. Print stats
tokens = train_data["token_count"]
avg_tokens = sum(tokens) / len(tokens) if tokens else 0
max_tokens = max(tokens) if tokens else 0

print(f"\n✅ Training Dataset Ready!")
print(f"   Total Samples: {len(train_data)}")
print(f"   Avg Tokens/Sample: {avg_tokens:.1f}")
print(f"   Max Tokens/Sample: {max_tokens}")


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/345 [00:00<?, ?B/s]

num_proc must be <= 3. Reducing num_proc to 3 for dataset of size 3.
[datasets.arrow_dataset|WARNING]num_proc must be <= 3. Reducing num_proc to 3 for dataset of size 3.


...

Filter (num_proc=3):   0%|          | 0/3 [00:00<?, ? examples/s]

num_proc must be <= 3. Reducing num_proc to 3 for dataset of size 3.
[datasets.arrow_dataset|WARNING]num_proc must be <= 3. Reducing num_proc to 3 for dataset of size 3.


Valid samples: 3 (Dropped 0)
Applying chat template using 4 cores...


Map (num_proc=3):   0%|          | 0/3 [00:00<?, ? examples/s]


✅ Training Dataset Ready!
   Total Samples: 3
   Avg Tokens/Sample: 20.0
   Max Tokens/Sample: 23


## 🏋️ Step 7 — Train

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

# With only a few samples the model needs many more epochs to converge.
# Rule of thumb for tiny datasets: aim for ~500-1000 total gradient steps.
# 4 samples, packing -> ~1 packed sequence -> 1 step/epoch
# => 300 epochs gives ~300 steps, enough for clear character imprinting.
trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = train_data,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = MAX_SEQ_LENGTH, # Ensure this is 2048+ for NCERT chapters
        packing = True,                  # Essential for efficient 500k token processing
        dataset_num_proc = 4,            # Faster preprocessing on Kaggle's CPU
        
        # --- Stability Settings ---
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 8,  # Increased for a smoother "Total Batch Size" of 16
        warmup_steps = 50,                # Longer warmup to prevent gradient spikes in the beginning
        
        # --- Traning Settings ---
        num_train_epochs = 56,             # 10 is too high for 1k+ convos; 3 ensures memory without "brain fry"
        learning_rate = 8e-5,             # Lowered for precise, deep learning of lore and facts
        lr_scheduler_type = "cosine",     # Best for natural character "settling"
        
        # --- Technical Rigor ---
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",             # Saves VRAM for the larger dataset
        weight_decay = 0.05,              # Higher decay prevents her from memorizing textbooks word-for-word
        seed = 3407,
        
        # --- Checkpointing ---
        output_dir = "/kaggle/working/nayari_checkpoints",
        save_steps = 50,                  # Save frequently to catch the "sweet spot" of the loss curve
        save_total_limit = 2,
        report_to = "none",
        ddp_find_unused_parameters = False,
    ),
)

gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name} | VRAM: {round(gpu.total_memory/1024**3,1)} GB")
stats = trainer.train()
print(f"\n✅ Done in {stats.metrics['train_runtime']:.0f}s")


num_proc must be <= 3. Reducing num_proc to 3 for dataset of size 3.
[datasets.arrow_dataset|WARNING]num_proc must be <= 3. Reducing num_proc to 3 for dataset of size 3.


Unsloth: Tokenizing ["text"] (num_proc=3):   0%|          | 0/3 [00:00<?, ? examples/s]

num_proc must be <= 3. Reducing num_proc to 3 for dataset of size 3.
[datasets.arrow_dataset|WARNING]num_proc must be <= 3. Reducing num_proc to 3 for dataset of size 3.


Unsloth: Packing train dataset (num_proc=3):   0%|          | 0/3 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!
GPU: Tesla T4 | VRAM: 14.6 GB


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3 | Num Epochs = 3 | Total steps = 3
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 275,214,336 of 2,052,302,336 (13.41% trained)


Step,Training Loss
1,1.111257
2,1.111257
3,1.111257


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:356: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")



✅ Done in 6s
